# Silver — Internal Events

Reads OLTP table families per operator and unifies them.

The internal side has three physically separate tables per operator:

| canonical txn_type | internal table family              |
|--------------------|------------------------------------|
| SUB_OK          | `sub_initial_{operator}`           |
| RECUR_OK        | `sub_recursion_success_{operator}` |
| RECUR_FAIL      | `sub_recursion_failure_{operator}` |


**Output**: silver.internal_events


## Parameters

In [13]:
import datetime as dt

ingestion_date = dt.date.today().isoformat()
ods_database = "ods"   # database where the per-operator tables live
silver_table = "silver_internal_events"

print(f"ingestion_date = {ingestion_date}")
print(f"ods_database = {ods_database}")
print(f"silver_table = {silver_table}")


StatementMeta(, 63e0ad0e-d0e1-41de-9ad2-d7b26955f19d, 15, Finished, Available, Finished, False)

ingestion_date = 2026-05-26
ods_database   = ods
silver_table   = silver_internal_events


## (One-time) Load sample data
CSVs from Files/Sample_Data/Internal/ are written to a Delta table.


In [14]:
import os

sample_dir = "/lakehouse/default/Files/Sample_Data/Internal"

if os.path.isdir(sample_dir):
    for fname in sorted(os.listdir(sample_dir)):
        if not fname.endswith(".csv"):
            continue
        table_name = f"{fname[:-4]}"   # strip .csv
        if spark.catalog.tableExists(table_name):
            print(f"  exists, skipping: {table_name}")
            continue
        df = (spark.read.option("header", "true").option("inferSchema", "true")
                       .csv(f"file:{sample_dir}/{fname}"))
        df.write.format("delta").saveAsTable(table_name)
        print(f"  created {table_name} ({df.count()} rows)")
else:
    print(f"No sample dir at {sample_dir} — assuming OLTP tables already populated")


StatementMeta(, 63e0ad0e-d0e1-41de-9ad2-d7b26955f19d, 16, Finished, Available, Finished, False)

  created sub_initial_telco_a (5 rows)
  created sub_initial_telco_b (3 rows)
  created sub_recursion_failure_telco_a (1 rows)
  created sub_recursion_success_telco_a (3 rows)
  created user_churn_events (1 rows)


## Canonical schema

In [15]:
from pyspark.sql import DataFrame, functions as F, types as T
from delta.tables import DeltaTable
from functools import reduce

INTERNAL_EVENT_SCHEMA = T.StructType([
    T.StructField("operator_code",T.StringType(),False),
    T.StructField("internal_id",T.StringType(),False),
    T.StructField("internal_id_type",T.StringType(),False),
    T.StructField("user_id",T.StringType(),False),
    T.StructField("partner_txn_id",T.StringType(),True),
    T.StructField("txn_type",T.StringType(),False),
    T.StructField("status",T.StringType(),True),
    T.StructField("amount",T.DecimalType(18, 4),True), 
    T.StructField("txn_ts_utc",T.TimestampType(),False),
    T.StructField("business_date",T.DateType(),False),
    T.StructField("ingestion_date",T.DateType(),False),
    T.StructField("source_table",T.StringType(),False),
    T.StructField("_ingested_at",T.TimestampType(),False),
])


StatementMeta(, 63e0ad0e-d0e1-41de-9ad2-d7b26955f19d, 17, Finished, Available, Finished, False)

## Dynamic discovery

We list every table in ODS, filter by the family prefix, and parse the operator code from the suffix.


In [31]:
def discover_tables(prefix: str) -> list[tuple[str, str]]:

    discovered = []
    for t in spark.catalog.listTables():
        if t.name.startswith(prefix):
            operator_code = t.name[len(prefix):]
            discovered.append((operator_code, f"{t.name}"))
    return sorted(discovered)

for prefix in ["sub_initial_", "sub_recursion_success_", "sub_recursion_failure_"]:
    found = discover_tables(prefix)
    print(f"{prefix}: {[op for op, _ in found]}")


StatementMeta(, 63e0ad0e-d0e1-41de-9ad2-d7b26955f19d, 33, Finished, Available, Finished, False)

sub_initial_: ['telco_a', 'telco_b']
sub_recursion_success_: ['telco_a']
sub_recursion_failure_: ['telco_a']


## Normalizers


In [17]:
def normalize_sub_initial(operator_code: str, table_name: str) -> DataFrame:
    raw = spark.table(table_name)
    return (raw
        .withColumn("operator_code",F.lit(operator_code))
        .withColumn("internal_id",F.col("sub_id"))
        .withColumn("internal_id_type",F.lit("subscription"))
        .withColumn("txn_type",F.lit("SUB_OK"))
        .withColumn("amount",F.col("amount").cast(T.DecimalType(18, 4)))
        .withColumn("txn_ts_utc",F.col("created_ts_utc").cast(T.TimestampType()))
        .withColumn("business_date",F.to_date(F.col("txn_ts_utc")))
        .withColumn("ingestion_date",F.lit(ingestion_date).cast(T.DateType()))
        .withColumn("source_table",F.lit(table_name))
        .withColumn("_ingested_at",F.current_timestamp())
        .select(*[f.name for f in INTERNAL_EVENT_SCHEMA.fields])
    )


StatementMeta(, 63e0ad0e-d0e1-41de-9ad2-d7b26955f19d, 19, Finished, Available, Finished, False)

sub_recursion_success (successful renewals)



In [18]:
def normalize_sub_recursion_success(operator_code: str, table_name: str) -> DataFrame:
    raw = spark.table(table_name)
    return (raw
        .withColumn("operator_code",     F.lit(operator_code))
        .withColumn("internal_id",       F.col("recursion_id"))
        .withColumn("internal_id_type",  F.lit("recursion_success"))
        .withColumn("txn_type",          F.lit("RECUR_OK"))
        .withColumn("status",            F.lit("n/a"))
        .withColumn("amount",            F.col("amount").cast(T.DecimalType(18, 4)))
        .withColumn("txn_ts_utc",        F.col("recurrence_ts_utc").cast(T.TimestampType()))
        .withColumn("business_date",     F.to_date(F.col("txn_ts_utc")))
        .withColumn("ingestion_date",    F.lit(ingestion_date).cast(T.DateType()))
        .withColumn("source_table",      F.lit(table_name))
        .withColumn("_ingested_at",      F.current_timestamp())
        .select(*[f.name for f in INTERNAL_EVENT_SCHEMA.fields])
    )


StatementMeta(, 63e0ad0e-d0e1-41de-9ad2-d7b26955f19d, 20, Finished, Available, Finished, False)

sub_recursion_failure (failed renewals)


In [19]:
def normalize_sub_recursion_failure(operator_code: str, table_name: str) -> DataFrame:
    raw = spark.table(table_name)
    return (raw
        .withColumn("operator_code",F.lit(operator_code))
        .withColumn("internal_id",F.col("failure_id"))
        .withColumn("internal_id_type",F.lit("recursion_failure"))
        .withColumn("txn_type",F.lit("RECUR_FAIL"))
        .withColumn("status",F.lit("n/a"))
        .withColumn("amount",F.lit(None).cast(T.DecimalType(18, 4)))
        .withColumn("txn_ts_utc",F.col("attempt_ts_utc").cast(T.TimestampType()))
        .withColumn("business_date",F.to_date(F.col("txn_ts_utc")))
        .withColumn("ingestion_date",F.lit(ingestion_date).cast(T.DateType()))
        .withColumn("source_table",F.lit(table_name))
        .withColumn("_ingested_at",F.current_timestamp())
        .select(*[f.name for f in INTERNAL_EVENT_SCHEMA.fields])
    )


StatementMeta(, 63e0ad0e-d0e1-41de-9ad2-d7b26955f19d, 21, Finished, Available, Finished, False)

## Normalizers Dict


In [20]:
TABLE_FAMILIES = [
    ("sub_initial_",            normalize_sub_initial),
    ("sub_recursion_success_",  normalize_sub_recursion_success),
    ("sub_recursion_failure_",  normalize_sub_recursion_failure),
]


StatementMeta(, 63e0ad0e-d0e1-41de-9ad2-d7b26955f19d, 22, Finished, Available, Finished, False)

## Idempotent MERGE

Match key is `(operator_code, internal_id)`


In [21]:
def merge_into_silver(new_events: DataFrame, table_name: str):
    if not spark.catalog.tableExists(table_name):
        (new_events.write
            .format("delta")
            .partitionBy("business_date", "operator_code")
            .saveAsTable(table_name))
        print(f"Created {table_name} with {new_events.count()} rows")
        return

    target = DeltaTable.forName(spark, table_name)
    (target.alias("t").merge(
        new_events.alias("s"),
        condition="t.operator_code = s.operator_code AND t.internal_id = s.internal_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"Merged into {table_name}")


StatementMeta(, 63e0ad0e-d0e1-41de-9ad2-d7b26955f19d, 23, Finished, Available, Finished, False)

## Dispatcher

for each normalizer, discover its operator tables, normalize each,union them and merge into silver.


In [25]:
def build_slv_internal_evt():
    all_frames = []
    for prefix, normalizer in TABLE_FAMILIES:
        discovered = discover_tables(prefix)
        if not discovered:
            print(f"  {prefix}*: no tables")
            continue

        print(f"  {prefix}*: {[op for op, _ in discovered]}")
        for operator_code, table_name in discovered:
            print(table_name)
            frame = normalizer(operator_code, table_name)
            all_frames.append(frame)

    if not all_frames:
        print("No internal tables discovered")
        return

    unified = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=False), all_frames)
    merge_into_silver(unified, silver_table)


StatementMeta(, 63e0ad0e-d0e1-41de-9ad2-d7b26955f19d, 27, Finished, Available, Finished, False)

## Run

In [32]:
build_slv_internal_evt()

StatementMeta(, 63e0ad0e-d0e1-41de-9ad2-d7b26955f19d, 34, Finished, Available, Finished, False)

  sub_initial_*: ['telco_a', 'telco_b']
sub_initial_telco_a
sub_initial_telco_b
  sub_recursion_success_*: ['telco_a']
sub_recursion_success_telco_a
  sub_recursion_failure_*: ['telco_a']
sub_recursion_failure_telco_a
Created silver_internal_events with 12 rows


## Sanity checks

In [33]:
# display(spark.sql(f"SELECT * FROM {silver_table} LIMIT 10"))


StatementMeta(, 63e0ad0e-d0e1-41de-9ad2-d7b26955f19d, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8952921d-2208-4d94-985d-4260e8dcd7a0)